[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/solutions/01_clinical_utility.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# From AF scores to clinical decisions

We have now trained ECG classifiers that are capable of assigning a score to each
recording. A service cannot act on a score. It needs to decide, based on the score
for each recording, whether to **send the recording for AF review** or not. This
practical will discuss that decision: the cost of different outcomes, how these
prices drive the determination of a threshold, and why changes in these prices
change which model or threshold is optimal.

At the end of this practical you should have produced, for a single classifier:

- a decision threshold from a priced decision matrix;
- a function for pricing any set of review decisions;
- a comparison of two models at the same prices, and how this changes with the price
  of missing an AF recording;
- a recalibration fit on its own split, and the net benefit for each model given the
  service's preference;
- a frozen policy prior to evaluation, the evaluation report including uncertainty
  intervals, and a written decision record.

## How to work

Please work through the notebook sequentially and on your own. Please note that each
problem is marked as either **Make** (you need to write a function or assign a
value) or **Fix the bug** (you need to debug an existing piece of code that runs but
produces an inaccurate result). Please replace each `...` with your solution and
then run the cell. A checker function is provided at the end of each problem cell to
verify your answer. The checker will provide one of three outputs: ✓, "Not yet" with
a hint as to the likely error, or "Not attempted yet" if the `...` has not been
replaced. The checker will never stop your notebook. Cells without a problem simply
run library code for you. Please inspect the output as we provide guidance on how to
interpret it below the cell. Collapsed explanations are best opened once you've
attempted the problem yourself. All widgets have an ordinary function call in the
subsequent cell that does the same job if you'd prefer not to use the widget.

**Stuck for about 10 minutes?** Move on. For functions that are required by later
cells, a commented line such as `# policy_cost = check.use_reference("policy_cost")`
will be provided below the function. Simply delete the `#` and run the cell to
proceed with a reference implementation of the function. If a subsequent cell
requires an answer you've not completed, it will print `Waiting for ...` with the
name of the problem that needs to be completed. Once this is completed or switched,
you can click inside the waiting cell and select **Runtime > Run before** and then
run the waiting cell. You can find worked versions of all notebooks in the
`solutions/` folder of the repository.

| Section | Time | Problems |
|---|---|---|
| 1. Price the decision | about 8 minutes | Problem 1 |
| 2. Six development recordings | about 10 minutes | Problem 2 |
| 3. The cost of a policy | about 7 minutes | Problem 3 |
| 4. Every operating point, and J | about 12 minutes | |
| 5. Which model? It depends on the price of a miss | about 12 minutes | Problem 4 |
| 6. Calibration and net benefit | about 14 minutes | Problems 5 and 6 |
| 7. Freeze, evaluate and record | about 12 minutes | Problem 7 |

Times are rough estimates and are provided as
guidance. **If you are behind, skip these** and come back to them later: the
explorer widget in Section 3, the resampling cell in Section 5, and the decision
curve widget in Section 6 (the static figure is all that is needed). Do not skip
Section 7.

### Notation

| Term | Meaning |
|---|---|
| review | Send the recording for AF review: a clinician reads the ECG. |
| `TP`, `FP`, `FN`, `TN` | Counts of recordings with each outcome: reviewed with AF, reviewed without AF, AF not reviewed (a miss), and neither. |
| $C_{TP}$, $C_{FP}$, $C_{FN}$, $C_{TN}$ | The price of one recording with that outcome, in reviews. In code, `costs["FN"]` holds $C_{FN}$. |
| loss | The sum of the prices over a set of decisions, in reviews. It is what we minimise to choose a policy; the results we report are reviews, AF detected and AF missed. |
| score | A model's output for one recording. It is a probability only if it is calibrated. |
| cutoff | Review when score ≥ cutoff. Functions call it `threshold`. |
| $p_t$ | The threshold probability: the AF probability at which both actions have the same expected loss. Decision curves call it the **preference**. |
| operating point | One cutoff and its consequences: the four counts, the rates and the loss. |
| per 1,000 | A count scaled to a thousand recordings of the split, so splits of different sizes compare directly. |

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

from clinical_utility import comparison, data, plots, reference, checks as check

costs = {"TP": 1, "FP": 1, "FN": 5, "TN": 0}
MODELS = ["random_forest", "resnet"]

## 1. Price the decision (about 8 minutes)

Our action is the decision to send a recording for AF review, where a clinician will
read the ECG. A positive decision is not a diagnosis or treatment decision. A
negative decision means the recording does not enter the review pathway, and is not
a definitive exclusion of AF for the individual.

If we cross our action with the reference label we have four potential outcomes,
with a price associated with each. We'll use a review as the unit of price and so
say that the price of a review is 1.

| | Reference AF | No reference AF |
|---|---:|---:|
| **Review** | $C_{TP}$ = 1 | $C_{FP}$ = 1 |
| **No review** | $C_{FN}$ = 5 | $C_{TN}$ = 0 |

A review is equivalently time consuming whether or not it finds AF, and so both
$C_{TP}$ and $C_{FP}$ have a price of one review. $C_{FP}$, a review that finds no
AF, has a total price of one review as we are assuming that there is no additional
harm done. If we miss a recording with AF we price this at 5 times as much as
sending it for review; we therefore save 4 if we send it for review (we still have
to pay 1 for the review). This is a value the service has determined; it is not an
intrinsic feature of the ECG. We have encoded this matrix in the `costs` dictionary
above.

### Before the outcome is known

Consider a recording with a probability of AF of $p$. We can find the expected loss
for each action by taking the weighted average of its two possible outcomes:

$$E[L(\text{review})] = C_{TP}\,p + C_{FP}\,(1-p), \qquad
E[L(\text{no review})] = C_{FN}\,p + C_{TN}\,(1-p).$$

If we consider a recording with a 10% probability of AF, the expected loss for
'review' would be 1 and for 'no review' would be 0.5. Therefore, we would select 'no
review'. If it was 60%, the expected loss for 'review' would be 1 and for 'no
review' would be 3, so we would select 'review'. We can see that there is a
threshold probability, $p_t$, where the expected loss for 'review' is the same as
for 'no review'. We would decide to review at this threshold as both would have the
same expected loss.

A key point here is that this framework requires a score that reflects an actual
probability: a score of 0.2 does not automatically reflect a 20% chance of the
recording having AF. We'll come back to this in Section 6.

In [ ]:
display(plots.expected_cost_figure(costs))

### Problem 1: Make

Complete `threshold_probability(costs)`. Solve for $p$ which makes the two expected
losses equal. We denote the additional loss of reviewing a recording without AF as
$\text{harm} = C_{FP} - C_{TN}$ and the loss saved by reviewing a recording with AF
as $\text{benefit} = C_{FN} - C_{TP}$, and thus:

$$p_t = \frac{\text{harm}}{\text{harm} + \text{benefit}}.$$

We can now use this to produce a rule to review if $p \ge p_t$; however, this
requires both harm and benefit to be positive. If this is not the case, the rule
does not apply and we need to consider the expected losses of the two options
directly, as for certain combinations of sign the optimal choice will either be the
same at every probability or will invert based on the underlying probability. Raise
a `ValueError`. The checker attempts to use the default matrix, followed by a matrix
with a more expensive miss, and finally a matrix where no two prices are the same.

In [ ]:
def threshold_probability(costs):
    """The AF probability at which review and no review have equal expected loss."""
    harm = costs["FP"] - costs["TN"]
    benefit = costs["FN"] - costs["TP"]
    if harm <= 0 or benefit <= 0:
        raise ValueError("The threshold rule needs C_FP > C_TN and C_FN > C_TP.")
    return harm / (harm + benefit)


_ = check.threshold_probability(threshold_probability)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# threshold_probability = check.use_reference("threshold_probability")

In [ ]:
if check.ready(threshold_probability=threshold_probability):
    p_t = threshold_probability(costs)
    miss_prices = [2, 5, 10, 15, 20, 30]
    display(pd.DataFrame({"price of a miss, in reviews": miss_prices,
                          "threshold probability": [threshold_probability({**costs, "FN": m})
                                                    for m in miss_prices]}).round(3))

The table can be read from top to bottom. If we increase the price of missing AF, we
increase the benefit of reviewing a recording and so reduce the threshold
probability. This means that we will send recordings for review at a lower level of
certainty. For our prices the threshold probability is 0.2. If the price of missing
AF was ten times the price of a review, the threshold would be 0.1. Note that the
absolute values of the prices do not matter, only the difference between $C_{FP}$
and $C_{TN}$ and between $C_{FN}$ and $C_{TP}$. If we add a constant to both
$C_{TP}$ and $C_{FN}$ it will not change the decision.

This can be tested using the *Missed AF* slider below, which sets $C_{FN}$.
The *FP harm* slider will add any additional harm of a false referral beyond the
review itself (e.g., anxiety or an unnecessary follow-up test), so that $C_{FP}$ =
1 + harm.

In [ ]:
expected_controls = plots.expected_cost_explorer()

In [ ]:
# Without widgets: change the two prices and rerun.
missed_af_price = 5
false_referral_harm = 0
display(plots.expected_cost_figure({"TP": 1, "FP": 1 + false_referral_harm,
                                    "FN": missed_af_price, "TN": 0}))

## 2. Six development recordings (about 10 minutes)

### The data

The 8,249 assessable recordings are the actual recordings from the
PhysioNet/CinC 2017 challenge that have the labels A (AF), N (normal rhythm) or O
(other rhythm), with their actual reference labels. The course scores for both
models are simulated for the purpose of teaching. While the O class is AF-negative
for this task, it is not considered a healthy rhythm. The 279 noisy recordings (`~`)
are not included in this practical as they do not have a binary AF reference; rather
than re-labelling them as normal, they are excluded. As the labels are for a full
recording, our decision will be at the recording level. The recordings have been
split into three groups as follows:

| Split | Recordings | With AF | Used for |
|---|---:|---:|---|
| Development | 3,711 | 341 | choosing policies |
| Calibration | 1,649 | 151 | fitting a recalibration, nothing else |
| Evaluation | 2,889 | 266 | assessing the policy you freeze in Section 7 |

In [ ]:
scores = data.load_scores()
development = scores[scores["split"] == "development"]
calibration_split = scores[scores["split"] == "calibration"]
evaluation = scores[scores["split"] == "evaluation"]
development.head()

We have a single row per recording with the recording ID, the reference label, `af`
(True if A), the dataset split, and one score per model. `resnet_calibrated` is the
ResNet model's score after undergoing a recalibration that you will fit yourself in
Section 6.

Six development recordings let you check every count by hand. These recordings were
selected to ensure that all outcomes are observed at both our $p_t$ and 0.5. They
include three AF recordings, a single O recording, and two N recordings.

In [ ]:
six = data.six_recordings()
six_decisions = six[["recording", "reference_label", "af", "random_forest"]].assign(
    review_at_p_t=six["random_forest"] >= 0.2,
    review_at_0_5=six["random_forest"] >= 0.5)
six_decisions

A cutoff of 0.5 corresponds to the default decision boundary of `model.predict()` of
a scikit-learn classifier (exact ties at 0.5 are classified as negative; the course
writes the cutoff explicitly as `predict_proba(X)[:, 1] >= 0.5`). We can contrast
this to our price-informed threshold of 0.2. We can see that the difference between
the two cutoffs is that there are two recordings out of six that would be reviewed
with a cutoff of 0.2 but not at a cutoff of 0.5. These recordings are the AF
recording A08248 and the N recording A00822.

### Problem 2: Fix the bug

A colleague has attempted to count the outcomes using scikit-learn. The code runs
without issue; however, the counts for the six recordings are incorrect and it fails
if given a set of recordings that are all without AF. Fix this code. Your first step
should be to manually count the outcomes for the six recordings from the table
above. At 0.2 there should be 2 TP, 2 FP, 1 FN and 1 TN.

In [ ]:
def six_counts(y, review):
    """TP, FP, FN and TN for one set of review decisions."""
    tn, fp, fn, tp = confusion_matrix(y, review, labels=[False, True]).ravel()
    return {"TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}


print("At 0.2:")
_ = check.show(six_counts, six["af"], six["random_forest"] >= 0.2)
_ = check.six_counts(six_counts)

#### What went wrong

The confusion matrix function in scikit-learn, `confusion_matrix(y_true, y_pred)`,
has the reference labels in the rows and decisions in the columns, with both in
False/True order. This produces a matrix of `[[TN, FP], [FN, TP]]`, with `.ravel()`
returning `tn, fp, fn, tp`. The colleague has unpacked this in the order that it is
typically taught and so swaps TP and TN. The second issue is that the confusion
matrix will only have a single cell if a single class is present, unless the
`labels=[False, True]` argument is passed. These are both issues that are seen in
real evaluation code. The issue with the ravel order alone is sufficient to turn a
sensible policy into a bad one. If we price the policies with the swapped counts,
the cheapest-looking forest policy on the development split would review 3,686 of
the 3,711 recordings.

### Four rates, four denominators

The cell below uses scikit-learn's `recall_score` and `precision_score`. For the
positive class (AF) this would be sensitivity and PPV; for `pos_label=False` this is
specificity and NPV. `zero_division=np.nan` ensures that rates with empty
denominators are undefined instead of zero.

| Rate | Question | Denominator |
|---|---|---|
| Sensitivity | Of the AF recordings, what fraction is reviewed? | TP + FN |
| Specificity | Of those without AF, what fraction avoids review? | TN + FP |
| PPV | Of the reviewed recordings, what fraction has AF? | TP + FP |
| NPV | Of the unreviewed recordings, what fraction has no AF? | TN + FN |

In [ ]:
display(pd.DataFrame({
    "six at 0.2": reference.operating_summary(six["af"], six["random_forest"] >= 0.2),
    "six at 0.5": reference.operating_summary(six["af"], six["random_forest"] >= 0.5),
    "development, nobody reviewed": reference.operating_summary(
        development["af"], np.zeros(len(development), dtype=bool)),
}).round(3))

Compare the columns to the table of six recordings above. At 0.2, we detect A08248
but A00822 (which does not have AF) is also reviewed: we have improved sensitivity
from 0.33 to 0.67 and reduced specificity from 0.67 to 0.33.

You can see a reporting trap in the last column. If we do not review anyone, the PPV
for AF is undefined and the NPV for AF is 0.908, which sounds great; however, we've
missed all 341 AF recordings in the development split. The reason the NPV is so high
is because AF is rare (9.2% of the split), not because we've found anything. You
should report the number of AF recordings missed alongside NPV. Sensitivity and
specificity are conditioned on the reference label; PPV and NPV are conditioned on
the decision, and so they are also prevalence sensitive.

## 3. The cost of a policy (about 7 minutes)

A policy is the decision for all recordings. The policy loss is the sum, over the
four outcomes, of the number of each outcome times its price.

### Problem 3: Make

Complete the function `policy_cost(y, review, costs)`. Use `confusion_matrix` (with
`labels=[False, True]`) to count outcomes in the same order as the fixed code from
Problem 2. Use values from `costs` for all prices so that the function works even if
the service has different prices for the outcomes. The checker will also test your
function with a matrix where no two prices are the same.

In [ ]:
def policy_cost(y, review, costs):
    """Total loss, in reviews, of one set of review decisions."""
    tn, fp, fn, tp = confusion_matrix(y, review, labels=[False, True]).ravel()
    return tp * costs["TP"] + fp * costs["FP"] + fn * costs["FN"] + tn * costs["TN"]


_ = check.policy_cost(policy_cost)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# policy_cost = check.use_reference("policy_cost")

In [ ]:
if check.ready(policy_cost=policy_cost):
    ledger = plots.show_policy(six, 0.2, costs, policy_cost)

This table is a ledger of results. We have a row for each recording with the outcome
and price. This is a transparent way of calculating the overall cost, as anyone can
check the results. At 0.2 we can see that the overall cost for these six recordings
is 9 reviews. This is 4 reviews and one missed AF recording which costs 5. At 0.5
this will cost 12. Note that these six recordings are simply to ensure that the
numbers are correct, not for choosing a cutoff. This will be done in Section 4 using
the development split. The explorer below prices the six recordings with your
implementation of `policy_cost`. It keeps the version it was given, so if you update
the function, you will need to re-run the explorer cell.

In [ ]:
if check.ready(policy_cost=policy_cost):
    teaching_controls = plots.policy_explorer(policy_cost, six)

In [ ]:
# Without widgets: change the cutoff or the prices and rerun.
if check.ready(policy_cost=policy_cost):
    _ = plots.show_policy(six, 0.5, {"TP": 1, "FP": 1, "FN": 5, "TN": 0}, policy_cost)

## 4. Every operating point, and J (about 12 minutes)

To find the optimal cutoff on the development split, we can look at all possible
cutoffs, one at every distinct score. An ordered list of all distinct scores,
highest first, can be obtained using
`roc_curve(y, scores, drop_intermediate=False)`. This will return every cutoff, with
the true-positive and false-positive rates at each. The cell below converts these to
the four counts as well as the loss for each cutoff. This is achieved by the
`reference.operating_points` function, which is a short block of about ten lines of
pandas around `roc_curve` that you can read in the course package. The result is
that the first row has the cutoff infinity, which reviews no one, and the last row
reviews everyone.

In [ ]:
points = reference.operating_points(development["af"], development["random_forest"], costs)
display(points.iloc[[0, 1, 2, -2, -1]].round(3))
print(len(points), "operating points")

In [ ]:
cheapest = points.sort_values(["loss", "reviews"], kind="stable").iloc[0]
max_j = points[points["youden_j"] == points["youden_j"].max()].sort_values("reviews").iloc[0]
n = len(development)
display(pd.DataFrame({
    label: {"cutoff": row["threshold"], "reviews per 1,000": 1000 * row["reviews"] / n,
            "AF detected per 1,000": 1000 * row["TP"] / n,
            "AF missed per 1,000": 1000 * row["FN"] / n, "J": row["youden_j"], "loss": row["loss"]}
    for label, row in (("minimum loss", cheapest), ("maximum J", max_j))}).round(3))
display(plots.objective_figure(points, {"Minimum loss": cheapest, "Maximum J": max_j}))

We can read the two columns as two policies. A common approach is to look for the
cutoff that maximises Youden's J, the sum of sensitivity and specificity minus one.
We can see that the optimal cutoff according to Youden's J would be at 0.102. In
contrast, our loss-minimising cutoff is 0.239. At J's cutoff, we would review 165.2
per 1,000 recordings compared to 110.5 per 1,000. This will result in a miss rate
of 14.0 per 1,000 instead of 21.8 per 1,000. On this split, this is an additional
203 reviews for an additional 29 true positives. However, the other 174 of those
reviews are not true positives. At our prices it is more expensive to make these
extra reviews than to miss the AF recordings they find, which is why the loss
chooses the higher cutoff. We break ties between minimum-loss cutoffs by selecting
the one with the fewest reviews.

### Why J disagrees

J weighs the two error *rates* equivalently. This means that, as there are
$n_{neg}/n_{pos}$ = 9.88 recordings without AF for each AF recording, every AF
recording that is missed is penalised as if 9.88 additional false referrals had been
made. This is the penalty that J will use for an incremental miss ($C_{FN} -
C_{TP}$), relative to a false referral ($C_{FP} - C_{TN}$). Our matrix sets the
penalty for an incremental miss at 4 times that for a false referral. If we set
$C_{TP}$ and $C_{FP}$ to 1 and $C_{TN}$ to 0, then J is effectively saying that a
miss should be priced as if it cost about 10.88 reviews; however, this price is not
based on anyone's judgement but rather on the case mix.

#### The algebra, and what J implies

Given that $TP = n_{pos} - FN$ and $TN = n_{neg} - FP$,

$$J = 1 - \frac{FN}{n_{pos}} - \frac{FP}{n_{neg}},$$

and the result of this is that maximising J is the same as minimising
$\frac{n_{neg}}{n_{pos}}FN + FP$. For our matrix, $L = n_{pos}C_{TP} + FP\,(C_{FP} -
C_{TN}) + FN\,(C_{FN} - C_{TP}) + n_{neg}C_{TN}$, and so minimising the loss is the
same as minimising $(C_{FN} - C_{TP})FN + (C_{FP} - C_{TN})FP$. This means that J
uses $(C_{FN} - C_{TP})/(C_{FP} - C_{TN}) = n_{neg}/n_{pos}$ and so has a threshold
probability equal to the prevalence (9.2%).

If we simply change the case mix of the dataset but keep the distribution of scores
per class the same, the same cutoff will be selected by J, as the sensitivity and
specificity will be identical at each cutoff. Despite this, the implicit price of a
miss will have changed, although nobody's view of a missed AF recording has changed.
With the prices held fixed, it is the cutoff that minimises loss that will change,
despite the same cutoff being selected by J.

## 5. Which model? It depends on the price of a miss (about 12 minutes)

The ResNet model has a slightly better AUC than the forest (0.934 vs 0.926) on the
development split. AUC is an average performance across all cutoffs and so does not
integrate the prices. For our use case, the service will use a single cutoff under a
fixed matrix.

### Problem 4: Fix the bug

A colleague attempts to compare the performance of the two models by reviewing any
recording with a score greater than or equal to 0.5 (the default boundary of
`predict()`). They produce a table that shows the number of reviews, AF detected,
and AF missed per 1,000 development recordings as well as the associated loss. From
these numbers, it seems as if the ResNet model is superior. Fix `compare_models` to
determine the minimum-loss cutoff for each model given the same prices (i.e.,
finding the cutoff that minimises loss from the model's operating-point table,
`reference.operating_points`, as in Section 4, resolving ties in loss by taking the
cutoff with the least number of reviews) and then produce the same table.

In [ ]:
def compare_models(frame, models, costs):
    """Each model's reviews, AF detected and AF missed per 1,000, and its loss."""
    y, n, rows = frame["af"], len(frame), []
    for model in models:
        table = reference.operating_points(y, frame[model], costs)
        cutoff = table.sort_values(["loss", "reviews"], kind="stable").iloc[0]["threshold"]
        review = frame[model] >= cutoff
        tn, fp, fn, tp = confusion_matrix(y, review, labels=[False, True]).ravel()
        rows.append({"model": model, "cutoff": cutoff, "reviews": 1000 * (tp + fp) / n,
                     "af_detected": 1000 * tp / n, "af_missed": 1000 * fn / n,
                     "loss": tp * costs["TP"] + fp * costs["FP"] + fn * costs["FN"] + tn * costs["TN"]})
    return pd.DataFrame(rows).set_index("model")


_ = check.show(compare_models, development, MODELS, costs)
_ = check.compare_models(compare_models)

#### What the shared cutoff hid

We can see from the results that when using a shared cutoff of 0.5, the raw ResNet
model has a lower loss (971) than the forest model (992) and detects more AF
recordings (319 vs 193). However, when using each model's own cheapest cutoff, the
forest model is actually cheaper (815 vs 922). This is because the two models
produce scores on a different scale, and so comparing them at a shared cutoff of 0.5
is not fair to either model. In addition, there is nothing magical about 0.5 as a
cutoff under these prices.

### The miss price decides

At each operating point, the number of reviews, AF detected and AF missed do not
depend on the prices; however, the overall loss does. The cell below recalculates
the loss for both models' operating points for a range of miss prices from 2 to 30
reviews and selects the cheapest policy for each model.

In [ ]:
sensitivity = reference.miss_price_sensitivity(development, MODELS, costs, range(2, 31))
wide = sensitivity.pivot(index="miss_price", columns="model", values="loss")
flip = wide.index[wide["resnet"] <= wide["random_forest"]][0]
display(sensitivity[sensitivity["miss_price"].isin([5, flip, 15])].round(3))
print("First miss price at which the raw ResNet is no dearer than the forest:", flip)
display(plots.model_choice_figure(sensitivity))

Read the table by miss price. We can see that for a miss price of 5 reviews, the
forest is cheaper and reviews 110.5 recordings per 1,000. When the price of a miss
increases to 15, the ResNet is cheaper. We can see it reviews 245.8 recordings per
1,000 and misses 3.8 per 1,000. The forest would review 165.2 recordings per 1,000
at the expense of 14.0 missed AF recordings per 1,000. We can see that the optimal
model changes as a function of the miss price. We can see that this flips when the
price of a miss is 8. This indicates that the model of choice cannot be determined
based on the better AUC, but rather by the price of a miss. The exact price of a
miss is a clinical and service judgement. This analysis demonstrates which judgement
the model choice depends on.

The specific value at which the model choice flips is based on this particular
development split, which has 341 AF recordings. The next cell resamples the
recordings with replacement while keeping both models' scores and the label for each
recording together (`scipy.stats.bootstrap` with `paired=True`), and finds the flip
in each resample. This will take a few seconds to run.

In [ ]:
flips = comparison.bootstrap_flip_prices(development, "random_forest", "resnet", costs,
                                         range(2, 31), resamples=200)
print(pd.Series(flips["flips"]).value_counts().sort_index().to_string())
print("Resamples with no flip up to 30:", flips["no_flip"])

We can see that in 90% of the 200 resamples, the flip price lies between 6 and 12.
This means that, based on one split, we can say that if the price of a miss is low,
we should use the forest, and if it is high, we should use the ResNet. We cannot say
that the exact price at which this flips is 8.

## 6. Calibration and net benefit (about 14 minutes)

If we use 0.2 as a threshold, we are implicitly assuming that the score is a
probability. We know that the raw ResNet model produces scores that are too high: in
the top decile of its development scores, the average score is 0.944 but only 48.7%
of those recordings actually have AF. Separately, we can see that the Brier score of
the raw ResNet model on the development split is 0.118 compared to 0.041 for the
forest model. The Brier score is the mean squared difference between the predicted
probability and the outcome; however, it is a composite metric of both calibration
as well as the model's ability to distinguish between AF and non-AF recordings and
so does not in isolation tell us in what direction the model is miscalibrated. We
can correct for this by constructing what's known as a Platt map. This fits a
regression model to predict the AF label given `logit(score)`. This requires access
to a set of recordings that is not used to select the policy, nor to evaluate the
policy. This is the purpose of the calibration split.

### Problem 5: Make

Complete `recalibration`. This should be a scikit-learn pipeline of
`FunctionTransformer(logit)` followed by
`LogisticRegression(C=np.inf, solver="newton-cholesky")` and fit on the `resnet`
scores (as a one-column DataFrame, `calibration_split[["resnet"]]`) and `af` labels
from the calibration split. `C=np.inf` is used to remove the default penalty that
`LogisticRegression` adds.

In [ ]:
recalibration = make_pipeline(FunctionTransformer(logit),
                              LogisticRegression(C=np.inf, solver="newton-cholesky")).fit(
    calibration_split[["resnet"]], calibration_split["af"])
_ = check.recalibration(recalibration)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# recalibration = check.use_reference("recalibration")

In [ ]:
if check.ready(recalibration=recalibration):
    recalibrated = recalibration.predict_proba(development[["resnet"]])[:, 1]
    y = development["af"]
    rows = {}
    for label, p in (("forest", development["random_forest"]), ("raw ResNet", development["resnet"]),
                     ("recalibrated ResNet", recalibrated)):
        summary = reference.operating_summary(y, p >= 0.2)
        rows[label] = {"AUC": roc_auc_score(y, p), "Brier": brier_score_loss(y, p),
                       "reviews per 1,000 at 0.2": 1000 * summary["reviews"] / len(y),
                       "AF missed per 1,000 at 0.2": 1000 * summary["af_missed"] / len(y)}
    display(pd.DataFrame(rows).round(3))

Look first at the row for AUC. You'll note that the recalibrated ResNet retains the
same AUC and the same best achievable policy as the raw ResNet, as the order of the
scores is maintained; however, what it means for a score to be 0.2 has changed. The
raw scores are too high and so at 0.2, the raw ResNet model will send 283.2
recordings per 1,000 for review, while the recalibrated model will send 199.1. The
Brier score of the model has reduced from 0.118 to 0.056. Note that a single Brier
score is not necessarily proof of calibration for all subgroups or even for a new
population. We've included an extension task (Extension 1) to produce reliability
plots as well as to calculate the calibration intercept and slope.

### Net benefit from the priced matrix

We can think about the value of a given policy by comparing it to the policy of
reviewing no one. If we review an AF recording we prevent a miss priced at 5 at the
cost of a review priced at 1. This means that for each correct review we gain 4. If
we review a recording without AF, we simply lose the price of a review, 1. We can
define the value of a policy, its loss avoided, as $4\,TP - FP$. We can then divide
by the value of a correct review, 4, so that the unit is the gain from one correct
review, as well as by the number of recordings $n$. This means that we can determine
the net benefit of a policy as follows:

$$NB(p_t) = \frac{TP}{n} - \frac{FP}{n}\cdot\frac{p_t}{1-p_t}.$$

We can see that the weight for false positives is the odds of the threshold
probability, in this case 0.25. This means that the net benefit is in true-positive
equivalents per recording, relative to the policy of reviewing no one. This is not a
count of strokes prevented but rather a relative value measure under the valuation
that $p_t$ encodes. Based on this, we can see that for the six recordings at a
threshold of 0.2, the net benefit would be NB = 2/6 − 2/6 × 0.25 = 0.250.

### Problem 6: Make

Complete the function `net_benefit(y, review, p_t)`.

In [ ]:
def net_benefit(y, review, p_t):
    """Net benefit per recording at threshold probability p_t, relative to reviewing nobody."""
    tn, fp, fn, tp = confusion_matrix(y, review, labels=[False, True]).ravel()
    n = tn + fp + fn + tp
    return tp / n - fp / n * p_t / (1 - p_t)


_ = check.net_benefit(net_benefit)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# net_benefit = check.use_reference("net_benefit")

In [ ]:
if check.ready(net_benefit=net_benefit, recalibration=recalibration):
    frame = development.assign(resnet_recalibrated=recalibration.predict_proba(
        development[["resnet"]])[:, 1])
    curve = reference.decision_curve(frame, ["random_forest", "resnet", "resnet_recalibrated"],
                                     np.round(np.arange(.01, .66, .01), 2), net_benefit)
    display(curve[curve["preference"].isin([0.1, 0.2, 0.4])].set_index("preference").round(4))
    display(plots.decision_curves(curve, 0.2))

The curves should be interpreted vertically. If we fix a particular preference
first, we can then compare the various strategies to one another at that value. We
cannot compare points across different preferences, as a different preference
corresponds to a different value system for the same outcomes, and so the highest
point across preferences means nothing. At a preference of 0.2, we can see that the
forest has a net benefit of 0.059, the recalibrated ResNet has a net benefit
of 0.051, and the raw ResNet has a net benefit of 0.041. The net benefit of
reviewing everyone would be -0.135 and, by definition, reviewing no one has a net
benefit of zero. We can see that the ResNet is improved by the recalibration;
however, the forest still has the highest net benefit at this preference.

If we set $C_{TP}$ and $C_{FP}$ to 1 and $C_{TN}$ to 0, then for a preference $p_t$,
$1/p_t$ is the price of a miss in units of reviews. We can see that the recalibrated
ResNet model is superior to the forest model up to a preference of about 0.12. This
is the point where the price of a miss is about 8.3 reviews. This is close to the
results from Section 5 where the optimal model flips. This is because both models'
stated probabilities are approximately calibrated, and so acting at a stated $p_t$
is approximately the same as acting at each model's best cutoff for that price.

In [ ]:
if check.ready(net_benefit=net_benefit):
    dca_controls = plots.decision_curve_explorer(
        lambda frame, models, preferences: reference.decision_curve(frame, models, preferences,
                                                                    net_benefit),
        development, ["random_forest", "resnet", "resnet_calibrated"])

In [ ]:
# Without widgets: change the preference and rerun.
preference = 0.2
if check.ready(net_benefit=net_benefit):
    curve = reference.decision_curve(development, ["random_forest", "resnet", "resnet_calibrated"],
                                     np.round(np.arange(.01, .66, .01), 2), net_benefit)
    display(plots.decision_curves(curve, preference))

## 7. Freeze, evaluate and record (about 12 minutes)

We have to decide on our model, cutoff and prices before looking at the evaluation
split, and then evaluate the frozen policy on it. The policy we've selected is the
forest model at its development minimum-loss cutoff. You may decide to use a
different policy, such as a cutoff derived from the prices instead. If you can
justify it, you can use it. The issue with selecting a cutoff on the development
split and then evaluating it on the same split is that the results are
only *apparent* results. The cutoff is selected as the best of many options on the
same recordings and so the results will be overly favourable. The only way to know
what the frozen policy does on recordings that played no part in choosing it is to
evaluate it on the evaluation split.

### Problem 7: Make

Please complete `frozen` with a dictionary containing the following keys: `model`
(one of `"random_forest"`, `"resnet"` or `"resnet_calibrated"`), `cutoff` (the
cutoff on that model's score scale), `costs` (the four prices), `p_t` (the threshold
probability those prices imply) and `chosen_on` (a brief description of how you
chose the cutoff). Once this is set, please run the reveal. Note that you can re-run
the reveal after changing your choice; however, this is not fresh validation but
rather repeatedly looking at the same data.

In [ ]:
frozen = {"model": "random_forest", "cutoff": float(cheapest["threshold"]),
          "costs": dict(costs), "p_t": threshold_probability(costs),
          "chosen_on": "development minimum loss at the default prices"}
frozen_ok = check.frozen_policy(frozen)

In [ ]:
# Reveal: the frozen policy on development (apparent) and on evaluation, with 95% intervals.
reports = {}
if not frozen_ok:
    check.waiting("Problem 7: freeze your policy above")
else:
    for name, frame in (("development", development), ("evaluation", evaluation)):
        review = frame[frozen["model"]] >= frozen["cutoff"]
        reports[name] = reference.evaluation_report(frame["af"], review, frozen["p_t"])
    reports["development"] = reports["development"][["estimate"]]
    _ = plots.show_report({"development (apparent)": reports["development"],
                           "evaluation": reports["evaluation"]})

The evaluation column shows the 95% intervals based on a paired percentile bootstrap
(`scipy.stats.bootstrap`). This is based on resampling the recordings with their
associated decisions. This is a measure of the sampling uncertainty for recordings
of this type. This is not a measure of the uncertainty introduced by the policy
selection process, and a held-out split from the same cohort says nothing about how
the policy may generalise to a different service.

#### The worked policy on evaluation

On the evaluation split, the worked policy results in 117.3 reviews per 1,000
recordings (95% interval 105.9 to 128.8) and misses 22.2 AF recordings per 1,000
(16.6 to 27.7). This is in contrast to the development split, where the policy
appeared to review 110.5 per 1,000 and miss 21.8 per 1,000. The net benefit of the
policy on the evaluation split is 0.058 (0.049 to 0.068) compared to the apparent
net benefit of 0.060.

### Problem 7, continued: the decision record

Please complete `decision_record`, the record a service would file. Fill in
`action`, `price_rationale` (your justification for why a miss is worth the price
you've set) and `reopen_if` (what would have to change for you to revisit this
decision) in a sentence or two each; the other fields come from your frozen policy
and the various reports. Consider the following before writing:

1. **The pathway.** Who is alerted if a recording is flagged, how quickly are they
   alerted, and what do they do next? If a recording is never reviewed, who is
   responsible? If nobody has capacity to act on a review, it has no benefit,
   whatever the matrix says.
2. **Drift and prevalence.** What happens if the AF rate goes up or down, or the
   recording device changes and the scores shift? What numbers from your report
   would you monitor to look for this? Extensions 1 and 4 work through recalibration
   and prevalence.
3. **What J assumed.** Maximising J implied a miss price set by the case mix. What
   price of missing an AF recording did it imply here, and why would this change if
   the prevalence changes?
4. **Evidence before use.** Which of these does today's analysis address: a
   prospective evaluation on local data with the local case mix and pathway;
   monitoring of how many reviews are being done, how many are being missed, and
   whether the model remains calibrated after go-live; clinical safety management,
   which in England would require the manufacturer to comply with DCB0129 and the
   organisation implementing it to comply with DCB0160; a regulatory assessment of
   whether the software is a medical device, which depends on its intended purpose
   (the MHRA regulates devices in the UK)?

In [ ]:
decision_record = {
    "action": "Send each recording whose forest score is at or above the cutoff for AF review "
              "by a clinician.",
    "price_rationale": "A review costs one unit of clinician time. A missed AF recording is "
                       "priced at five reviews because undetected AF forgoes anticoagulation; "
                       "at that price review is worthwhile above an AF probability of 0.2.",
    "reopen_if": "Reopen the decision if the service prices a miss above about eight reviews, "
                 "where the ResNet becomes cheaper, or if prevalence, the recording device or "
                 "the review capacity changes.",
}
if not reports:
    check.waiting("Problem 7: freeze your policy and run the reveal above")
else:
    _ = check.decision_record(decision_record, frozen, reports)

## Continue with the extensions

Please find the extension tasks within the `extension/` folder. Each extension task
leverages the same matrix and the course's reference implementations, and as such
they are not contingent on the answers you've provided here.

| | Extension | The question it answers | Time |
|---|---|---|---|
| 1 | Calibration: Platt, isotonic and prior shift | Can a score be read as an AF probability, and which recalibration should you trust? | about 35 minutes |
| 2 | Comparing models: reclassification and net-benefit differences | Which recordings change decision between two models, and does the change help? | about 30 minutes |
| 3 | Class weighting versus moving the threshold | Does reweighting the training loss do anything a different cutoff cannot? | about 30 minutes |
| 4 | Capacity, prevalence and daily queues | What happens when AF is rarer, or the review team can only see so many recordings a day? | about 30 minutes |
| 5 | Window-level calls and unreadable recordings | How should window scores become one recording-level call, and what should happen to a recording nobody can read? | about 35 minutes |
| 6 | Your own classifier's scores | What does today's analysis say about the classifier you trained this week? | about 25 minutes |

If you found Section 6 hard, please attempt Extension 1. If you're interested in a
more nuanced model comparison, please attempt Extension 2. If you're interested in
the implications for model deployment, please attempt Extension 4. If you have a set
of scores from your own classifier, please attempt Extension 6.